# 02 — Built-up Classification via NDBI

**Goal:** turn the proven Nairobi composite (notebook 01) into a binary built-up/non-built-up classification, and measure how good it actually is against an independent reference (ESA WorldCover) rather than eyeballing it.

Baseline approach: NDBI (Normalized Difference Built-up Index) thresholded at 0 — the simplest possible classifier. This establishes a real, measured baseline number before any tuning or a trained model is considered.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary, get_sentinel2_composite

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
composite, scene_count = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
print(f'Composite ready: {scene_count} scenes')

Composite ready: 11 scenes


## NDBI and threshold

`NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)`, using Sentinel-2 bands B11 (SWIR1) and B8 (NIR). Built-up surfaces (concrete, roofing, bare ground) reflect relatively more SWIR than NIR, so they trend toward higher NDBI; vegetation and water trend lower. `ee.Image.normalizedDifference(['B11', 'B8'])` computes this directly.

Threshold: NDBI > 0. This is the textbook default, not tuned to Nairobi specifically — the point of this notebook is to measure how well that untuned default actually performs.

In [2]:
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndbi_builtup = ndbi.gt(0).rename('builtup')

## Reference: ESA WorldCover

ESA WorldCover 2021 (`ESA/WorldCover/v200`, 10m resolution) is an independently produced, published global land-cover map — not ground truth, but a real external reference rather than a number we made up. Class value 50 in its `Map` band is "Built-up." We pull the same class for Nairobi and compare pixel-by-pixel against our NDBI classification.

In [3]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(nairobi)
worldcover_builtup = worldcover.eq(50).rename('builtup')

agreement_image = ndbi_builtup.eq(worldcover_builtup)
agreement_stats = agreement_image.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=nairobi,
    scale=10,
    maxPixels=1e9,
)
agreement_pct = agreement_stats.getInfo()['builtup'] * 100
print(f'Pixel agreement with WorldCover built-up class: {agreement_pct:.1f}%')

Pixel agreement with WorldCover built-up class: 62.0%


## Diagnosing the disagreement

A single agreement number doesn't say *how* the two methods disagree. Checking each method's built-up area fraction separately shows whether NDBI is systematically over- or under-classifying relative to WorldCover, which is more actionable than the agreement percentage alone.

In [4]:
ndbi_frac = ndbi_builtup.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

wc_frac = worldcover_builtup.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

print(f'NDBI-classified built-up: {ndbi_frac:.1f}% of Nairobi')
print(f'WorldCover-classified built-up: {wc_frac:.1f}% of Nairobi')

NDBI-classified built-up: 56.4% of Nairobi
WorldCover-classified built-up: 31.9% of Nairobi


## Visualize

Interactive comparison map (live in Jupyter), plus a static thumbnail of the NDBI classification alone for a quick no-Jupyter check.

In [5]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'True color', False)
Map.addLayer(ndbi_builtup, builtup_vis, 'NDBI built-up (this notebook)')
Map.addLayer(worldcover_builtup, builtup_vis, 'WorldCover built-up (reference)', False)
Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [6]:
for name, image in [('ndbi', ndbi_builtup), ('worldcover', worldcover_builtup)]:
    url = image.getThumbURL({
        'min': 0, 'max': 1, 'palette': ['black', 'red'],
        'region': nairobi, 'dimensions': 800,
    })
    path = f'../data/processed/nairobi_builtup_{name}.png'
    urllib.request.urlretrieve(url, path)
    print(f'Saved {path}')

Saved ../data/processed/nairobi_builtup_ndbi.png


Saved ../data/processed/nairobi_builtup_worldcover.png


## Iteration: combining NDBI with NDVI

Diagnosis from the thumbnails above: Nairobi National Park (the southeast wedge) is largely misclassified as built-up by NDBI alone. Dry-season bare soil and dry grassland have a spectral signature — low vegetation reflectance relative to SWIR — that looks a lot like concrete/roofing to a single index. NDBI can't tell "no vegetation because it's a building" from "no vegetation because it's dry savanna."

Fix being tested: require **both** NDBI > 0 **and** NDVI < 0.2 (a common textbook low-vegetation cutoff, also untuned) before calling a pixel built-up. `NDVI = (NIR - Red) / (NIR + Red)`, i.e. `normalizedDifference(['B8', 'B4'])`. This should reject anything with even moderate vegetation cover — including most dry grassland — while keeping true concrete/roofing, which has near-zero NDVI regardless of season.

In [7]:
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
combined_builtup = ndbi.gt(0).And(ndvi.lt(0.2)).rename('builtup')

combined_agreement = combined_builtup.eq(worldcover_builtup).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

combined_frac = combined_builtup.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

print(f'NDBI+NDVI agreement with WorldCover: {combined_agreement:.1f}%  (was {agreement_pct:.1f}% with NDBI alone)')
print(f'NDBI+NDVI built-up fraction: {combined_frac:.1f}% of Nairobi  (was {ndbi_frac:.1f}% with NDBI alone, WorldCover says {wc_frac:.1f}%)')

NDBI+NDVI agreement with WorldCover: 81.7%  (was 62.0% with NDBI alone)
NDBI+NDVI built-up fraction: 17.8% of Nairobi  (was 56.4% with NDBI alone, WorldCover says 31.9%)


In [8]:
url = combined_builtup.getThumbURL({
    'min': 0, 'max': 1, 'palette': ['black', 'red'],
    'region': nairobi, 'dimensions': 800,
})
path = '../data/processed/nairobi_builtup_ndbi_ndvi_combined.png'
urllib.request.urlretrieve(url, path)
print(f'Saved {path}')

Saved ../data/processed/nairobi_builtup_ndbi_ndvi_combined.png


## Sweeping the NDVI threshold

Rather than guess a single new cutoff, sweep a range and measure both agreement and built-up fraction at each — the fraction is currently under WorldCover's, so we expect a looser (higher) NDVI cutoff to let more true built-up pixels back in, but by how much before agreement stops improving is an empirical question, not something to assume.

In [9]:
ndvi_thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
sweep_results = []

for t in ndvi_thresholds:
    candidate = ndbi.gt(0).And(ndvi.lt(t)).rename('builtup')
    agreement = candidate.eq(worldcover_builtup).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
    ).getInfo()['builtup'] * 100
    frac = candidate.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
    ).getInfo()['builtup'] * 100
    sweep_results.append({'ndvi_threshold': t, 'agreement_pct': agreement, 'builtup_frac_pct': frac})
    print(f'NDVI < {t:.2f}  ->  agreement {agreement:.1f}%,  built-up fraction {frac:.1f}%  (WorldCover: {wc_frac:.1f}%)')

NDVI < 0.15  ->  agreement 78.9%,  built-up fraction 13.4%  (WorldCover: 31.9%)


NDVI < 0.20  ->  agreement 81.7%,  built-up fraction 17.8%  (WorldCover: 31.9%)


NDVI < 0.25  ->  agreement 83.5%,  built-up fraction 22.0%  (WorldCover: 31.9%)


NDVI < 0.30  ->  agreement 83.6%,  built-up fraction 26.6%  (WorldCover: 31.9%)


NDVI < 0.35  ->  agreement 81.2%,  built-up fraction 32.6%  (WorldCover: 31.9%)


NDVI < 0.40  ->  agreement 76.5%,  built-up fraction 39.8%  (WorldCover: 31.9%)


In [10]:
best = max(sweep_results, key=lambda r: r['agreement_pct'])
best_threshold = best['ndvi_threshold']
print(f"Best NDVI threshold by agreement: {best_threshold:.2f}  ({best['agreement_pct']:.1f}% agreement, {best['builtup_frac_pct']:.1f}% built-up fraction)")

final_builtup = ndbi.gt(0).And(ndvi.lt(best_threshold)).rename('builtup')

url = final_builtup.getThumbURL({
    'min': 0, 'max': 1, 'palette': ['black', 'red'],
    'region': nairobi, 'dimensions': 800,
})
path = f'../data/processed/nairobi_builtup_ndvi_{best_threshold:.2f}.png'
urllib.request.urlretrieve(url, path)
print(f'Saved {path}')

Best NDVI threshold by agreement: 0.30  (83.6% agreement, 26.6% built-up fraction)


Saved ../data/processed/nairobi_builtup_ndvi_0.30.png
